In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import joblib

print("1. Cargando los datasets...")
# Leer los archivos CSV
df1 = pd.read_csv('cvs.csv')
df2 = pd.read_csv('Curriculum Vitae.csv')

# Unir ambos datasets para tener más data de entrenamiento
df = pd.concat([df1, df2], ignore_index=True)
# Eliminar posibles filas duplicadas o vacías
df = df.dropna().drop_duplicates()
print(f"Total de currículos cargados: {len(df)}")

print("2. Limpiando el texto de los currículos...")
def limpiar_texto(texto):
    # Convertir a minúsculas
    texto = texto.lower()
    # Eliminar URLs
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    # Eliminar caracteres especiales y números (dejar solo letras)
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    # Eliminar espacios extra
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# Aplicar la limpieza a la columna Resume
df['Resume_Limpio'] = df['Resume'].apply(limpiar_texto)

print("3. Vectorizando el texto y codificando categorías...")
# TfidfVectorizer convierte el texto a una matriz numérica basada en la frecuencia de las palabras
# (Guardaremos esto como un .pkl)
vectorizador = TfidfVectorizer(max_features=3000)
X = vectorizador.fit_transform(df['Resume_Limpio']).toarray()

# LabelEncoder convierte las categorías de texto (ej. "Data Science") a números (ej. 0, 1, 2)
# (Guardaremos esto como otro .pkl)
encoder = LabelEncoder()
y = encoder.fit_transform(df['Category'])
num_clases = len(np.unique(y))

# Dividir la data: 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("4. Construyendo la Red Neuronal con TensorFlow/Keras...")
model = Sequential()
# Capa de entrada
model.add(Dense(512, activation='relu', input_dim=X.shape[1]))
model.add(Dropout(0.5)) # Dropout para evitar el sobreajuste (overfitting)
# Capa oculta
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
# Capa de salida (Softmax para clasificación multiclase)
model.add(Dense(num_clases, activation='softmax'))

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("5. Entrenando el modelo...")
# Entrenar el modelo
historial = model.fit(X_train, y_train,
                      epochs=15,
                      batch_size=32,
                      validation_data=(X_test, y_test))

print("6. Exportando modelo y herramientas...")
# Exportar modelo Keras
model.save('modelo_clasificador_cv.h5')

# Exportar herramientas de Scikit-Learn
joblib.dump(vectorizador, 'vectorizador_cv.pkl')
joblib.dump(encoder, 'encoder_categorias_cv.pkl')

print("¡Proceso Finalizado! Ya puedes descargar tus archivos .h5 y .pkl de la carpeta de Colab.")

1. Cargando los datasets...
Total de currículos cargados: 332
2. Limpiando el texto de los currículos...
3. Vectorizando el texto y codificando categorías...
4. Construyendo la Red Neuronal con TensorFlow/Keras...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5. Entrenando el modelo...
Epoch 1/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 76ms/step - accuracy: 0.1245 - loss: 3.1995 - val_accuracy: 0.4179 - val_loss: 3.1599
Epoch 2/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.3623 - loss: 3.0965 - val_accuracy: 0.2836 - val_loss: 3.0647
Epoch 3/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3925 - loss: 2.9455 - val_accuracy: 0.4030 - val_loss: 2.8851
Epoch 4/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.4981 - loss: 2.6550 - val_accuracy: 0.4328 - val_loss: 2.6216
Epoch 5/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5736 - loss: 2.2740 - val_accuracy: 0.5373 - val_loss: 2.2640
Epoch 6/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.6830 - loss: 1.8032 - val_accuracy: 0.7015 - val_loss: 1.8160
Epoch 7/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.8491 - loss: 1.3069 - val_accuracy: 0.8507 - val_loss: 1.3756
Epoch 8/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9170 - loss: 0.9038 - val_accuracy:

6. Exportando modelo y herramientas...
¡Proceso Finalizado! Ya puedes descargar tus archivos .h5 y .pkl de la carpeta de Colab.
